# Zillow leads workflow: run, dedupe, filter, export

A realistic lead-gen pass using the [Zillow Leads & Property Data](https://apify.com/germane_binoculars/zillow-leads-property-data) actor:

1. Run a `custom_search` order for a target metro, enriched depth
2. Track the `dedup_update` so a re-run next week never re-charges for rows you already have
3. Filter down to rows that actually have a callable agent phone number
4. Export a clean CSV ready for a CRM import

Requires `pip install apify-client pandas` and `APIFY_TOKEN` set in your environment.

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
from apify_client import ApifyClient

ACTOR_ID = "germane_binoculars/zillow-leads-property-data"
DEDUP_FILE = Path("dedup_state.json")  # persists across runs so you never re-pay for a row you already have

token = os.environ["APIFY_TOKEN"]
client = ApifyClient(token)

## 1. Load whatever dedup state exists from a prior run

First run: this file doesn't exist yet, so we start with empty lists.

In [ ]:
if DEDUP_FILE.exists():
    dedup_state = json.loads(DEDUP_FILE.read_text())
else:
    dedup_state = {"zpids": [], "mls_ids": []}

print(f"Already have {len(dedup_state['zpids'])} zpids on file")

## 2. Run a custom_search order

Swap in your own bounding box or `metro` name. `minListings`/`minEnriched` are floors,
not caps — you may get a bit more than requested, never less (short of the
`timeoutSecs` budget running out first).

In [ ]:
run_input = {
    "mode": "custom_search",
    "metro": "Phoenix",
    "depth": "enriched",
    "minListings": 200,
    "minEnriched": 100,
    "dedupZpids": dedup_state["zpids"],
    "dedupMlsIds": dedup_state["mls_ids"],
    "timeoutSecs": 3600,
}

run = client.actor(ACTOR_ID).call(run_input=run_input, wait_secs=3600)
print(f"Run {run['id']} finished: {run['status']}")

## 3. Pull the rows and update dedup state

The `dedup_update` record lives in the run's key-value store (not the dataset,
so it never shows up as a stray non-listing row if you're iterating dataset
items programmatically elsewhere).

In [ ]:
dataset_id = run["defaultDatasetId"]
rows = list(client.dataset(dataset_id).iterate_items())
df = pd.DataFrame(rows)
print(f"Got {len(df)} rows")

kvs_id = run["defaultKeyValueStoreId"]
dedup_record = client.key_value_store(kvs_id).get_record("DEDUP_UPDATE")
if dedup_record:
    dedup_state = dedup_record["value"]
    DEDUP_FILE.write_text(json.dumps(dedup_state))
    print(f"Dedup state updated: now tracking {len(dedup_state['zpids'])} zpids")

## 4. Filter to rows with a callable agent phone, export a clean CSV

In [ ]:
leads = df[df["agent_phone"].notna()].copy()
print(f"{len(leads)} of {len(df)} rows have a callable agent phone number")

cols = [
    "address_street", "address_city", "address_state", "address_zipcode",
    "price", "bedrooms", "bathrooms", "living_area",
    "agent_name", "agent_phone", "agent_email", "broker_name",
    "days_on_zillow", "is_foreclosure", "detail_url",
]
leads = leads[[c for c in cols if c in leads.columns]]
leads.to_csv("phoenix_leads.csv", index=False)
leads.head()

## Next run

Run this notebook again next week: `dedup_state.json` already has this batch's
zpids/MLS IDs, so the next order only pays for genuinely new or re-listed rows.